# LG 지원 페이지 크롤러 — 인라인 이미지 태깅 버전

**변경 사항 (원본 lg_cr (1).ipynb 대비)**
- `li.txt-idt` 단위로 텍스트와 바로 아래 이미지를 쌍으로 수집
- 본문에 `[IMG:파일명.jpg]` 태그를 인라인으로 삽입
- 이미지는 이미 Supabase Storage에 저장되어 있으므로 **재다운로드 없음**
- CSV: `본문` 컬럼에 `[IMG:]` 태그 포함, `이미지목록`은 하위 호환용으로 유지

In [ ]:
import os
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# ── 카테고리 리스트 (원본과 동일) ──────────────────────────────────────────
categories = [
    # 냉장고/김치냉장고 (CT50019441)
    {"name": "양문형 냉장고", "query": "category=CT50019441&subCategory=CT50019468"},

    # 에어컨/환기 (CT50019183)
    {"name": "2in1 에어컨", "query": "category=CT50019183&subCategory=CT50019184"},
    {"name": "벽걸이형 에어컨", "query": "category=CT50019183&subCategory=CT50019214"},
    {"name": "가정용 천장형 에어컨", "query": "category=CT50019183&subCategory=CT50172000"},
    {"name": "상업용 천장형 에어컨", "query": "category=CT50019183&subCategory=CT50019259"},
    {"name": "상업용 스탠드 에어컨", "query": "category=CT50019183&subCategory=CT50019244"},
    {"name": "이동식 에어컨", "query": "category=CT50019183&subCategory=CT50019229"},
    {"name": "창호형 에어컨", "query": "category=CT50019183&subCategory=CT50108006"},
    {"name": "가정용 환기시스템", "query": "category=CT50019183&subCategory=CT50311000"},
    {"name": "상업용 환기시스템", "query": "category=CT50019183&subCategory=CT50311001"},

    # 세탁기/건조기/의류관리기 (CT50019274)
    {"name": "드럼세탁기", "query": "category=CT50019274&subCategory=CT50019309"},
    {"name": "통돌이", "query": "category=CT50019274&subCategory=CT50019324"},
    {"name": "워시타워", "query": "category=CT50019274&subCategory=CT50109011"},
    {"name": "미니세탁기", "query": "category=CT50019274&subCategory=CT50109013"},
    {"name": "의류건조기", "query": "category=CT50019274&subCategory=CT50019275"},
    {"name": "의류관리기", "query": "category=CT50019274&subCategory=CT50019292"},
    {"name": "시스템 아이어닝", "query": "category=CT50019274&subCategory=CT50366001"},
    {"name": "슈케이스", "query": "category=CT50019274&subCategory=CT50109016"},
    {"name": "슈케어", "query": "category=CT50019274&subCategory=CT50109017"},

    # TV / 프로젝터 (CT50019021)
    {"name": "올레드 TV", "query": "category=CT50019021&subCategory=CT50019037"},
    {"name": "QNED TV", "query": "category=CT50019021&subCategory=CT50109019"},
    {"name": "울트라 HD TV", "query": "category=CT50019021&subCategory=CT50019082"},
    {"name": "일반 LED TV", "query": "category=CT50019021&subCategory=CT50019052"},
    {"name": "시네빔", "query": "category=CT50019021&subCategory=CT50019022"},
    {"name": "나노셀 TV", "query": "category=CT50019021&subCategory=CT50109023"},
    {"name": "라이프 스타일 스크린", "query": "category=CT50019021&subCategory=CT50109024"},
    {"name": "일반형(TV)", "query": "category=CT50019021&subCategory=CT50109025"},
    {"name": "프로빔 (상업용)", "query": "category=CT50019021&subCategory=CT50109026"},

    # 노트북/모니터 (CT50019563)
    {"name": "그램/노트북", "query": "category=CT50019563&subCategory=CT50019564"},
    {"name": "일체형/데스크톱", "query": "category=CT50019563&subCategory=CT50019585"},
    {"name": "PC 모니터", "query": "category=CT50019563&subCategory=CT50019602"},
    {"name": "TV 모니터", "query": "category=CT50019563&subCategory=CT50109030"},
    {"name": "태블릿", "query": "category=CT50019563&subCategory=CT50019141"},

    # 주방가전 (CT50019695)
    {"name": "식기세척기", "query": "category=CT50019695&subCategory=CT50019735"},
    {"name": "정수기", "query": "category=CT50019695&subCategory=CT50019648"},
    {"name": "전자레인지", "query": "category=CT50019695&subCategory=CT50019709"},
    {"name": "전기레인지/인덕션", "query": "category=CT50019695&subCategory=CT50019761"},
    {"name": "광파오븐레인지", "query": "category=CT50019695&subCategory=CT50019696"},
    {"name": "가스오븐레인지", "query": "category=CT50019695&subCategory=CT50019722"},
    {"name": "가스레인지", "query": "category=CT50019695&subCategory=CT50019748"},
    {"name": "맥주제조기", "query": "category=CT50019695&subCategory=CT50019774"},

    # 청소기 (CT50019339)
    {"name": "무선청소기", "query": "category=CT50019339&subCategory=CT50019400"},
    {"name": "유선청소기", "query": "category=CT50019339&subCategory=CT50019380"},
    {"name": "로봇청소기", "query": "category=CT50019339&subCategory=CT50019340"},

    # 에어케어 (CT50019872)
    {"name": "공기청정기", "query": "category=CT50019872&subCategory=CT50019873"},
    {"name": "휴대용 공기청정기", "query": "category=CT50019872&subCategory=CT50019888"},
    {"name": "제습기", "query": "category=CT50019872&subCategory=CT50019890"},
    {"name": "가습기", "query": "category=CT50019872&subCategory=CT50019904"},
    {"name": "에어로타워", "query": "category=CT50019872&subCategory=CT50111012"},
    {"name": "에어로퍼니처", "query": "category=CT50019872&subCategory=CT50111013"},
    {"name": "전자식 마스크", "query": "category=CT50019872&subCategory=CT50111014"},
    {"name": "실링팬", "query": "category=CT50019872&subCategory=CT50019889"},

    # 뷰티/의료기기 (CT50019920)
    {"name": "클렌징 기기", "query": "category=CT50019920&subCategory=CT50019921"},
    {"name": "메디헤어", "query": "category=CT50019920&subCategory=CT50019932"},
    {"name": "아이케어", "query": "category=CT50019920&subCategory=CT50111017"},
    {"name": "탄력기기", "query": "category=CT50019920&subCategory=CT50111018"},
    {"name": "흡수촉진기", "query": "category=CT50019920&subCategory=CT50111019"},
    {"name": "메디페인", "query": "category=CT50019920&subCategory=CT50111020"},
]

print(f'카테고리 수: {len(categories)}')

In [ ]:
# ── 헬퍼 함수 ─────────────────────────────────────────────────────────────

def extract_file_id(src_url: str) -> str | None:
    """
    LG gscs 이미지 URL에서 파일명을 추출.
    예) https://gscs.lge.com/.../downloadFile.do?fileId=V74ff7Q...&portalId=P1
        → 'V74ff7Q....jpg'
    """
    if not src_url:
        return None
    try:
        params = parse_qs(urlparse(src_url).query)
        if 'fileId' in params:
            return params['fileId'][0] + '.jpg'
        # fallback: URL 경로의 마지막 세그먼트
        basename = os.path.basename(urlparse(src_url).path)
        return basename if len(basename) > 4 else None
    except:
        return None


def extract_inline_content(html: str) -> tuple[str, list[str]]:
    """
    innerHTML을 파싱하여 (인라인태깅본문, 이미지파일명목록)을 반환.

    1순위: li.txt-idt 단위로 텍스트 + 바로 아래 div.img-only img 를 쌍으로 처리
    2순위: fallback — 전체 텍스트 + 이미지 목록 (li.txt-idt 없는 페이지용)

    출력 예:
      '2. 설치 장소의 온도가 25도일 때... [IMG:V74ff7Q.jpg]\n3. 문을 닫고...'
    """
    soup = BeautifulSoup(html, 'html.parser')
    lines = []
    all_images = []

    li_items = soup.find_all('li', class_='txt-idt')

    if li_items:
        # ── li.txt-idt 단위 처리 ──────────────────────────────────────────
        for li in li_items:
            # 1) 이미지 추출 (div.img-only 안의 img)
            img_names = []
            for img_div in li.find_all('div', class_='img-only'):
                img_tag = img_div.find('img')
                if img_tag and img_tag.get('src'):
                    fname = extract_file_id(img_tag['src'])
                    if fname:
                        img_names.append(fname)
                        all_images.append(fname)
                img_div.decompose()  # 텍스트 추출 전 제거

            # 2) 텍스트 추출
            text = li.get_text(separator=' ', strip=True)
            if not text:
                continue

            # 3) 인라인 태깅: 텍스트 뒤에 [IMG:파일명] 추가
            if img_names:
                tag_str = ' '.join(f'[IMG:{n}]' for n in img_names)
                lines.append(f'{text} {tag_str}')
            else:
                lines.append(text)

    else:
        # ── Fallback: li.txt-idt 없는 페이지 ─────────────────────────────
        # img-only 이미지만 먼저 수집 후 제거
        for img_div in soup.find_all('div', class_='img-only'):
            img_tag = img_div.find('img')
            if img_tag and img_tag.get('src'):
                fname = extract_file_id(img_tag['src'])
                if fname:
                    all_images.append(fname)
            img_div.decompose()

        raw_text = soup.get_text(separator='\n', strip=True)
        lines = [t for t in raw_text.split('\n') if t.strip()]

    return '\n'.join(lines), all_images


print('헬퍼 함수 로드 완료')

In [ ]:
# ── Chrome 드라이버 설정 ───────────────────────────────────────────────────
options = webdriver.ChromeOptions()
options.add_experimental_option('excludeSwitches', ['enable-logging'])
options.add_argument('--headless')       # 백그라운드 실행
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')
options.add_argument('--window-size=1920,1080')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

output_folder = 'lg_total_crawl_tag'
os.makedirs(output_folder, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_filename = os.path.join(output_folder, f'lg_tag_results_{timestamp}.csv')

print(f'저장 파일: {csv_filename}')

In [ ]:
# ── 메인 크롤링 루프 ───────────────────────────────────────────────────────

for cat in categories:
    cat_name = cat['name']
    cat_query = cat['query']

    print(f'\n======== [{cat_name}] 수집 시작 ========')

    # 1단계: 카테고리 페이지에서 상세 URL 수집 (1~15페이지)
    cat_links = []
    for p in range(1, 16):
        base_url = f'https://www.lge.co.kr/support/solutions?{cat_query}&sort=update&page={p}'
        try:
            driver.get(base_url)
            time.sleep(4)
            links = driver.find_elements(By.CSS_SELECTOR, 'div.solution-result-list > ul > li a')
            if not links:
                break
            for link in links:
                url = link.get_attribute('href')
                if url and 'solutions' in url:
                    cat_links.append(url)
            print(f'   {cat_name} - {p}페이지 URL {len(links)}개 수집')
        except:
            break

    cat_links = list(set(cat_links))
    print(f'--- {cat_name} 총 {len(cat_links)}개 상세페이지 방문 시작 ---')

    # 2단계: 상세 페이지 수집
    for idx, target_url in enumerate(cat_links):
        try:
            driver.get(target_url)
            time.sleep(5)

            title = driver.find_element(By.CSS_SELECTOR, 'div.board-view-head > h2').text

            # 본문 영역 탐색 (다중 셀렉터)
            content_area = None
            for selector in ['.content-service-view', '#viewContent > div', '#viewContent']:
                try:
                    el = driver.find_element(By.CSS_SELECTOR, selector)
                    if el.is_displayed():
                        content_area = el
                        break
                except:
                    continue

            inline_body = ''
            all_image_names = []

            if content_area:
                html = content_area.get_attribute('innerHTML')
                inline_body, all_image_names = extract_inline_content(html)

            # CSV 행 저장 (이미지 다운로드 없음 — 스토리지에 이미 존재)
            new_row = {
                '카테고리': cat_name,
                '제목': title,
                '본문': inline_body,                           # [IMG:파일명] 인라인 태깅
                '이미지목록': ', '.join(all_image_names),      # 하위 호환용
                'URL': target_url,
            }

            df_row = pd.DataFrame([new_row])
            if not os.path.exists(csv_filename):
                df_row.to_csv(csv_filename, index=False, encoding='utf-8-sig')
            else:
                df_row.to_csv(csv_filename, index=False, encoding='utf-8-sig', mode='a', header=False)

            img_count = len(all_image_names)
            print(f'   ({idx+1}/{len(cat_links)}) 저장: {title[:20]}... (이미지 {img_count}개 태깅)')

        except Exception as e:
            print(f'   ! 에러: {target_url} — {e}')
            continue

print(f'\n✅ 모든 카테고리 수집 완료! 파일: {csv_filename}')
driver.quit()

---
## Pinecone 검색 테스트

현재 Pinecone에 있는 기존 데이터로 검색이 잘 되는지, 청크별 이미지가 잘 매핑되는지 확인.

> **참고**: 현재 Pinecone 데이터는 문서 전체 이미지 목록을 공유하는 구조임.
> 위 크롤러로 재수집 후 재업로드하면 청크별 이미지 1:1 매핑이 됨.

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone

load_dotenv(Path('../openaiapi.env'))  # 프로젝트 루트 기준

SUPABASE_URL = os.getenv('SUPABASE_URL', '').rstrip('/')
BUCKET = 'support-images'

oai = OpenAI()
pc  = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
idx = pc.Index(os.getenv('PINECONE_INDEX_NAME', 'lg-support'))

def make_image_url(filename: str) -> str:
    return f'{SUPABASE_URL}/storage/v1/object/public/{BUCKET}/{filename}'

def search_pinecone(query: str, top_k: int = 3):
    emb = oai.embeddings.create(model='text-embedding-3-small', input=[query]).data[0].embedding
    return idx.query(vector=emb, top_k=top_k, include_metadata=True)['matches']

print('Pinecone 연결 완료')
stats = idx.describe_index_stats()
print(f'벡터 수: {stats.total_vector_count}, 차원: {stats.dimension}')

In [ ]:
# ── 테스트 1: 냉장고 에러코드 ─────────────────────────────────────────────
TEST_QUERIES = [
    '냉장고 CH 에러코드',
    '세탁기 진동이 심해요',
    '에어컨 물이 떨어져요',
]

for query in TEST_QUERIES:
    print(f'\n{"="*60}')
    print(f'🔍 쿼리: "{query}"')
    print('='*60)

    matches = search_pinecone(query, top_k=3)

    for rank, m in enumerate(matches, 1):
        meta = m['metadata']
        raw_imgs = meta.get('image_urls', '')
        filenames = json.loads(raw_imgs) if raw_imgs else []
        image_urls = [make_image_url(f) for f in filenames]

        print(f'\n  [{rank}위] score={m["score"]:.3f}')
        print(f'  제목   : {meta.get("title", "")}')
        print(f'  기기   : {meta.get("device", "")}')
        print(f'  내용   : {meta.get("content_chunk", "")[:120]}...')
        print(f'  이미지 수: {len(image_urls)}개')
        for url in image_urls:
            print(f'    → {url}')

In [ ]:
# ── 테스트 2: 이미지-텍스트 매핑 시뮬레이션 ─────────────────────────────
# 현재 데이터 구조 vs 재크롤링 후 구조 비교

query = '냉장고 라벨 색이 변해요'
matches = search_pinecone(query, top_k=2)

print(f'🔍 쿼리: "{query}"\n')

for rank, m in enumerate(matches, 1):
    meta = m['metadata']
    raw_imgs = meta.get('image_urls', '')
    filenames = json.loads(raw_imgs) if raw_imgs else []

    content = meta.get('content_chunk', '')

    print(f'[{rank}위] {meta.get("title", "")} (score={m["score"]:.3f})')
    print()

    # 현재 구조: 텍스트 줄 + 이미지는 마지막에
    print('  ▼ 현재 구조 (이미지 분리):')
    for line in content.split('\n')[:5]:
        if line.strip():
            print(f'    {line.strip()}')
    if filenames:
        print(f'  이미지: {filenames}')  # 문서 전체 이미지 한꺼번에
    print()

    # 재크롤링 후 구조: [IMG:파일명] 인라인
    print('  ▼ 재크롤링 후 목표 구조 (인라인 태깅):')
    # 시뮬레이션: 파일을 하나씩 각 단락에 붙임
    lines = [l.strip() for l in content.split('\n') if l.strip()][:5]
    for i, line in enumerate(lines):
        img_tag = f' [IMG:{filenames[i]}]' if i < len(filenames) else ''
        print(f'    {line}{img_tag}')
    print()

In [ ]:
# ── 테스트 3: 이미지 URL 실제 접근 가능 여부 확인 ──────────────────────
import requests

query = '에어컨 실외기 누수'
matches = search_pinecone(query, top_k=1)

if matches:
    meta = matches[0]['metadata']
    raw_imgs = meta.get('image_urls', '')
    filenames = json.loads(raw_imgs) if raw_imgs else []

    print(f'제목: {meta.get("title", "")}')
    print(f'이미지 {len(filenames)}개 접근 테스트:')

    for fname in filenames[:3]:
        url = make_image_url(fname)
        try:
            r = requests.head(url, timeout=5)
            status = '✅ 접근 가능' if r.status_code == 200 else f'⚠️ {r.status_code}'
        except Exception as e:
            status = f'❌ 에러: {e}'
        print(f'  {status} → {url}')